In [0]:
# Kafka connection details (bootstrap servers, api key, api secret, topic)
# are retrieved at runtime from the Databricks secret scope 'finguard-scope',
# key 'kafka_connection_details' — see the next cell. Never hardcode them here.

In [0]:
# =====================================================
# Get Secret from the Databricks Secret Scope
# =====================================================
import json
try:
    kafka_connection  = dbutils.secrets.get(
        scope="finguard-scope",
        key='kafka_connection_details'
    )
    
    print("Secret retrieved successfully.")
    
    parsed = json.loads(kafka_connection)
    print("Parsed JSON:")
    print(parsed)
    
except Exception as e:
    print("Secret verification failed:")
    print(str(e))




In [0]:
# =====================================================
# Extract Kafka connection details from parsed JSON
# =====================================================
bootstrap_servers = parsed["bootstrap_servers"]
api_key = parsed["api_key"]
api_secret = parsed["api_secret"]
topic = parsed["topic"]

print("bootstrap_servers:", bootstrap_servers)
print("api_key:", api_key)
print("topic:", topic)
print("All values extracted successfully.")

In [0]:
jaas_config=f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{api_key}" password="{api_secret}";'


In [0]:
sample_batch=(spark.read.format("kafka")
              .option("kafka.bootstrap.servers",bootstrap_servers)
              .option("subscribe",topic)
            .option("kafka.security.protocol", "SASL_SSL")
        .option("kafka.sasl.mechanism", "PLAIN")
        .option("kafka.sasl.jaas.config", jaas_config)
        .option("startingOffsets","earliest")
        .load()
)


In [0]:
sample_batch.count()

In [0]:
sample_batch.display()

In [0]:
from pyspark.sql.functions import col
parsed_batch=sample_batch.select(
col("key").cast("string"),
col("value").cast("string"),
col("topic"),
col("partition"),
col("offset"),
col("timestamp"),
col("timestampType")
)



In [0]:
parsed_batch.display(20)

In [0]:
parsed_batch.write.saveAsTable("finguard.bronze.transactions_batch_test")

In [0]:
#Read Streaming of Data
streaming_df=(spark.readStream.format("kafka")
              .option("kafka.bootstrap.servers",bootstrap_servers)
              .option("subscribe",topic)
            .option("kafka.security.protocol", "SASL_SSL")
        .option("kafka.sasl.mechanism", "PLAIN")
        .option("kafka.sasl.jaas.config", jaas_config)
        .option("startingOffsets","earliest")
        .load()
)



In [0]:
from pyspark.sql.functions import col
parsed_streaming_df=streaming_df.select(
col("key").cast("string"),
col("value").cast("string"),
col("topic"),
col("partition"),
col("offset"),
col("timestamp"),
col("timestampType")
)

In [0]:
streaming_query=(parsed_streaming_df.writeStream.format("delta")
.outputMode("Append")
.option("checkpointLocation", "/Volumes/finguard/bronze/transacation_vol/checkpoint/")
.trigger(availableNow=True)
.toTable("finguard.bronze.transactions_streaming_test")
)
print("Query Started with query id: ",streaming_query.id)

In [0]:
%sql
select count(*) from finguard.bronze.transactions_streaming_test